In [21]:
import csv
import json
import shutil

from pathlib import Path

ENGINEER = "Olermo"
STUDENT_ID = "TUPM-26-1925"
SEED_DIGIT = 5

# Student-specific rules
TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_BEHAVIOR_CLASS = (SEED_DIGIT % 5) + 1

PROJECT_NAME = "Lab6_File_Handling_Additional_Exercises"

cwd = Path.cwd()

if cwd.name == PROJECT_NAME:
    PROJECT_DIR = cwd
elif cwd.name.startswith("Exercise") and cwd.parent.name == PROJECT_NAME:
    PROJECT_DIR = cwd.parent
elif (cwd / PROJECT_NAME / "data").is_dir():
    PROJECT_DIR = cwd / PROJECT_NAME
else:
    PROJECT_DIR = next(
        (p for p in cwd.parents if p.name == PROJECT_NAME),
        None
    )

if PROJECT_DIR is None:
    raise FileNotFoundError("Project folder not found.")

EXERCISE_DIR = PROJECT_DIR / "Exercise2_Mobile_Device"

DATA_FILE = PROJECT_DIR / "data" / "user_behavior_dataset.csv"

OUTPUT_DIR = EXERCISE_DIR / "output"
BACKUP_DIR = EXERCISE_DIR / "backup"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATA_FILE)
print("Dataset exists:", DATA_FILE.exists())

print("Target Operating System:", TARGET_OS)
print("Target Behavior Class:", TARGET_BEHAVIOR_CLASS)

Dataset: d:\Downloads\OLERMO\Lab6_File_Handling_Olermo\Lab6_File_Handling_Additional_Exercises\data\user_behavior_dataset.csv
Dataset exists: True
Target Operating System: iOS
Target Behavior Class: 1


In [11]:
import re

def normalize_column(name):
    return re.sub(r"[^a-z0-9]", "", name.lower())


def find_column(headers, possible_names):

    normalized = {
        normalize_column(header): header
        for header in headers
    }

    for name in possible_names:

        key = normalize_column(name)

        if key in normalized:
            return normalized[key]

    raise ValueError(
        f"Column not found: {possible_names}"
    )


def convert_number(value):

    try:
        return float(str(value).replace(",", "").strip())

    except (ValueError, TypeError):
        return None

In [12]:
selected_records = []

total_records = 0
matching_os_records = 0

battery_values = []
data_usage_values = []

try:

    with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as file:

        reader = csv.DictReader(file)

        headers = reader.fieldnames

        user_col = find_column(headers, ["User ID"])
        device_col = find_column(headers, ["Device Model"])
        os_col = find_column(headers, ["Operating System"])
        battery_col = find_column(
            headers, ["Battery Drain (mAh/day)", "Battery Drain"]
        )
        data_col = find_column(
            headers, ["Data Usage (MB/day)", "Data Usage"]
        )
        behavior_col = find_column(
            headers, ["User Behavior Class", "Behavior Class"]
        )

        for row in reader:

            total_records += 1

            operating_system = str(
                row.get(os_col, "")
            ).strip()

            if operating_system.lower() != TARGET_OS.lower():
                continue

            matching_os_records += 1

            behavior = convert_number(row.get(behavior_col))

            if behavior != TARGET_BEHAVIOR_CLASS:
                continue

            selected_records.append({
                user_col: row[user_col],
                device_col: row[device_col],
                os_col: row[os_col],
                battery_col: row[battery_col],
                data_col: row[data_col]
            })

            battery = convert_number(row.get(battery_col))
            usage = convert_number(row.get(data_col))

            if battery is not None:
                battery_values.append(battery)

            if usage is not None:
                data_usage_values.append(usage)

except FileNotFoundError:
    print("Dataset not found:", DATA_FILE)

except Exception as error:
    print("Error:", error)

In [13]:
average_battery = (
    sum(battery_values) / len(battery_values)
    if battery_values else None
)

average_data_usage = (
    sum(data_usage_values) / len(data_usage_values)
    if data_usage_values else None
)

print("Total records:", total_records)
print("Records matching OS:", matching_os_records)
print("Selected users:", len(selected_records))
print("Average battery drain:", average_battery)
print("Average data usage:", average_data_usage)

Total records: 700
Records matching OS: 146
Selected users: 22
Average battery drain: 467.22727272727275
Average data usage: 201.45454545454547


In [14]:
exported_csv = OUTPUT_DIR / "selected_mobile_users.csv"

export_fields = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)"
]

# Use the original header names discovered in the dataset.
actual_fields = [
    user_col,
    device_col,
    os_col,
    battery_col,
    data_col
]

with open(exported_csv, "w", encoding="utf-8", newline="") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=actual_fields
    )

    writer.writeheader()
    writer.writerows(selected_records)

print("Selected users exported successfully.")

Selected users exported successfully.


In [15]:
verified_count = 0

with open(exported_csv, "r", encoding="utf-8-sig", newline="") as file:

    reader = csv.DictReader(file)

    for row in reader:
        verified_count += 1

print("Selected record count:", len(selected_records))
print("Exported record count:", verified_count)

if verified_count == len(selected_records):
    print("Count verification successful.")
else:
    print("Count verification failed.")

Selected record count: 22
Exported record count: 22
Count verification successful.


In [16]:
summary = {
    "student": ENGINEER,
    "student_id": STUDENT_ID,
    "operating_system": TARGET_OS,
    "behavior_class": TARGET_BEHAVIOR_CLASS,
    "total_records": total_records,
    "records_matching_os": matching_os_records,
    "selected_records": len(selected_records),
    "average_battery_drain": average_battery,
    "average_data_usage": average_data_usage,
    "exported_record_count": verified_count,
    "count_verified": verified_count == len(selected_records)
}

json_file = OUTPUT_DIR / "mobile_usage_summary.json"

with open(json_file, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON summary created.")

JSON summary created.


In [17]:
backup_file = BACKUP_DIR / "selected_mobile_users_backup.csv"

shutil.copy2(exported_csv, backup_file)

if backup_file.exists():

    file_size = backup_file.stat().st_size

    print("Backup created successfully.")
    print("Backup file size:", file_size, "bytes")

else:
    print("Backup creation failed.")

Backup created successfully.
Backup file size: 674 bytes


In [18]:
print("\n===== MOBILE DEVICE USAGE PROFILING =====")

print("Operating System:", TARGET_OS)
print("Behavior Class:", TARGET_BEHAVIOR_CLASS)
print("Total records:", total_records)
print("Records matching OS:", matching_os_records)
print("Records in target behavior class:", len(selected_records))
print("Average battery drain:", average_battery)
print("Average data usage:", average_data_usage)
print("Count verification:", verified_count == len(selected_records))
print("Exported CSV:", exported_csv)
print("JSON summary:", json_file)
print("Backup file size:", backup_file.stat().st_size, "bytes")


===== MOBILE DEVICE USAGE PROFILING =====
Operating System: iOS
Behavior Class: 1
Total records: 700
Records matching OS: 146
Records in target behavior class: 22
Average battery drain: 467.22727272727275
Average data usage: 201.45454545454547
Count verification: True
Exported CSV: output\selected_mobile_users.csv
JSON summary: output\mobile_usage_summary.json
Backup file size: 674 bytes
